| # |  |  |
|---|----------|---------------|
| 1 | Regras de matching semânticas no prompt | Peeters & Bizer (2023) — regras em linguagem natural alcançam +5.94% F1 com custo mínimo |
| 3 | Pré-normalização do título antes do LLM | Ditto (Li et al., 2020) — span normalization reduz carga cognitiva do modelo |
| 4 | Batch prompting (múltiplos títulos por chamada) | BATCHER (Fan et al., 2024) — 4-7× redução de custo com qualidade equivalente |

**Arquitetura em 5 passos:**
1. Ingestão e Amostragem
2. Pré-Normalização Simbólica do Título (NOVO — Ditto-inspired)
3. Extração via LLM com Batch Prompting (prompt melhorado)
4. Limpeza Determinística (Camada Simbólica)
5. Exportação + Avaliação Estruturada

## Setup

In [1]:
!pip install -q tqdm google-generativeai python-dotenv

In [2]:
import os
import re
import json
import time
import logging
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

/Users/MezzowTecnologia/miniforge3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/tn/k0s30kj51dndyjqcmxlygffw0000gn/T/ipykernel_98811/1651798084.py:11: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [19]:
# ── Configuração ──────────────────────────────────────────────
DATASET_PATH  = "data/train.csv"
OUTPUT_PATH   = "output/resultado_pipeline_v2.csv"
SAMPLE_SIZE   = 200
RANDOM_SEED   = 42
BATCH_SIZE    = 5          # MELHORIA 4 — BATCHER (Fan et al., 2024)
API_KEY       = os.getenv("GOOGLE_API_KEY")
LLM_MODEL     = "gemini-3.1-flash-lite-preview"
TEMPERATURE   = 0.0

## Passo 1 — Ingestão e Amostragem

In [4]:
dataframe = pd.read_csv(DATASET_PATH)
df = dataframe[dataframe["language"] == "portuguese"].copy()
print(f"Total de produtos em português: {len(df)}")
df.head(5)

Total de produtos em português: 10000000


,title,label_quality,language,category
2,Maquina De Lavar Electrolux 12 Kilos,unreliable,portuguese,WASHING_MACHINES
3,Par Disco De Freio Diant Vent Gol 8v 08/ Frema...,unreliable,portuguese,VEHICLE_BRAKE_DISCS
5,"4 Microaspersor Irrigação Ultra 7,20 Metros",unreliable,portuguese,IRRIGATION_SPRINKLERS
6,Raquete Clash 100 Tour - Nova,unreliable,portuguese,RACQUETS
7,"Kit Tripe Para Celular Ou Câmera Fotog 1,20m +...",unreliable,portuguese,CAMERA_TRIPODS


In [5]:
df_sample = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)
print(f"Amostra: {len(df_sample)} registros")
print(f"Categorias presentes: {df_sample['category'].nunique()}")
df_sample["category"].value_counts().head(10)

Amostra: 200 registros
Categorias presentes: 187


category
AUTOMOTIVE_TIRES              2
FOOTBALL_CAPS                 2
WETSUITS                      2
MERCHANDISER_REFRIGERATORS    2
CAMERA_FLASHES                2
SANDWICH_MAKERS               2
HABERDASHERY_RIBBONS          2
MALE_UNDERWEAR                2
MARKING_AND_WARNING_TAPES     2
MOTORCYCLE_HANDLEBAR_YOKES    2
Name: count, dtype: int64

## Passo 2 — Pré-Normalização do Título (MELHORIA 3)

> **Fundamentação:** Ditto (Li et al., 2020) demonstrou que *span normalization* —
> reescrever spans equivalentes antes de alimentar o modelo — melhora
> significativamente o matching. A ideia: se o LLM recebe "500ml" em vez de
> "1/2 litro", a carga cognitiva é menor e a saída mais consistente.

In [6]:
# ── Pré-normalização do título (Ditto-inspired) ───────────────
# Aplicada ANTES do LLM para reduzir variação na entrada.
# A camada simbólica PÓS-LLM continua existindo como safety net.

_PRE_RULES = [
    # Frações de litro → ml
    (re.compile(r"1/2\s*l(?:itro)?s?\b", re.I),          "500ml"),
    (re.compile(r"meio\s+litro", re.I),                   "500ml"),
    (re.compile(r"(\d+)[,.]5\s*l(?:itro)?s?\b", re.I),
        lambda m: f"{int(m.group(1))*1000+500}ml"),
    (re.compile(r"(\d+)\s*l(?:itro)?s?\b", re.I),
        lambda m: f"{int(m.group(1))*1000}ml"),

    # Unidades coladas / variantes
    (re.compile(r"(\d+)\s*k(?:ilo)?s?\b", re.I),
        lambda m: f"{m.group(1)}kg"),
    (re.compile(r"(\d+)\s*(?:gramas?|gr)\b", re.I),
        lambda m: f"{m.group(1)}g"),

    # Voltagem
    (re.compile(r"bi[\s-]?volt(?:s)?\b", re.I),          "bivolt"),
    (re.compile(r"(\d+)\s*volt?s?\b", re.I),
        lambda m: f"{m.group(1)}v"),

    # Espaços excessivos
    (re.compile(r"\s{2,}"), " "),
]


def pre_normalize_title(title: str) -> str:
    """Aplica regras de normalização no título ANTES de enviar ao LLM."""
    text = title.strip()
    for pattern, replacement in _PRE_RULES:
        if callable(replacement):
            text = pattern.sub(replacement, text)
        else:
            text = pattern.sub(replacement, text)
    return text.strip()


# ── Teste rápido ──
test_cases = [
    "Maquina De Lavar Electrolux 12 Kilos",
    "Garrafa Térmica 1/2 Litro Aço Inox",
    "Liquidificador Mondial Turbo 110 Volts",
    "Coca Cola meio litro",
    "Panela de Pressão 4,5 Litros",
    "Cerveja Heineken 600 ml Pack 6",
]
print("Pré-normalização (Ditto-inspired):")
print("-" * 65)
for t in test_cases:
    print(f"  {t:<45s} → {pre_normalize_title(t)}")

Pré-normalização (Ditto-inspired):
-----------------------------------------------------------------
  Maquina De Lavar Electrolux 12 Kilos          → Maquina De Lavar Electrolux 12kg
  Garrafa Térmica 1/2 Litro Aço Inox            → Garrafa Térmica 500ml Aço Inox
  Liquidificador Mondial Turbo 110 Volts        → Liquidificador Mondial Turbo 110v
  Coca Cola meio litro                          → Coca Cola 500ml
  Panela de Pressão 4,5 Litros                  → Panela de Pressão 4500ml
  Cerveja Heineken 600 ml Pack 6                → Cerveja Heineken 600 ml Pack 6


In [7]:
# Aplicar pré-normalização à amostra
df_sample["titulo_pre_normalizado"] = df_sample["title"].apply(pre_normalize_title)

# Quantificar impacto
alterados = (df_sample["title"] != df_sample["titulo_pre_normalizado"]).sum()
print(f"Títulos alterados pela pré-normalização: {alterados}/{len(df_sample)} ({alterados/len(df_sample)*100:.1f}%)")
df_sample[df_sample["title"] != df_sample["titulo_pre_normalizado"]][["title", "titulo_pre_normalizado"]].head(10)

Títulos alterados pela pré-normalização: 47/200 (23.5%)


,title,titulo_pre_normalizado
1,Ferro De Passar Dimar Elétrico Funcionando Rar...,Ferro De Passar Dimar Elétrico Funcionando Rar...
5,Coxim Diant Motor L200 4x4 95/ Mobensani ...,Coxim Diant Motor L200 4x4 95/ Mobensani
8,Café Poesia Torrado Em Grão 1 Kg .............,Café Poesia Torrado Em Grão 1 Kg ............ ...
13,Conjunto Social Infantil Pajem Camisa Colete ...,Conjunto Social Infantil Pajem Camisa Colete G...
22,Araras Expositor Rt De Painel 10 Unidades Branco,Araras Expositor Rt De Painel 10 Unidades Branco
23,Controle Remoto + Suporte Ar Condicionado Lg O...,Controle Remoto + Suporte Ar Condicionado Lg O...
24,Fogão Semi Industrial Uma Boca,Fogão Semi Industrial Uma Boca
25,Kit 4 Cola Epóxi Liquido Transparente 10min 1...,Kit 4 Cola Epóxi Liquido Transparente 10min 16...
26,Furadeira 1/2 Impacto 127v Gsb 550 Re+14broca...,Furadeira 1/2 Impacto 127v Gsb 550 Re+14brocas...
28,Cimento Queimado Quartzolit - Azul Urbano - 4...,"Cimento Queimado Quartzolit - Azul Urbano - 4,..."


## Passo 3 — Extração via LLM (Melhorias 1 e 4)

### Melhoria 1 — Regras de Matching Semânticas (Peeters & Bizer, 2023)
> Regras em linguagem natural no prompt alcançaram 88.29% F1 (+5.94%) com custo
> apenas 2× do zero-shot. Adicionamos regras de **desambiguação semântica**.

### Melhoria 4 — Batch Prompting (BATCHER, Fan et al., 2024)
> Batch prompting agrupa múltiplos títulos em uma chamada, cortando custo
> de API em 4-7×.

In [8]:
SYSTEM_PROMPT = """
Você é um especialista em extração de atributos de produtos de e-commerce brasileiro.

══════════════════════════════════════════════════════════════════
TAREFA
══════════════════════════════════════════════════════════════════
Você receberá um ou mais títulos de produtos, cada um identificado por um
índice numérico. Para CADA título, retorne um objeto JSON com as chaves
abaixo. Retorne um ARRAY JSON (lista) contendo um objeto por título, na
mesma ordem recebida. Se receber apenas 1 título, retorne um array com 1
objeto.

Chaves obrigatórias (use null se o atributo não existir no título):
  "idx"                : int    — o índice recebido
  "marca"              : string — ex: "Electrolux", "Samsung", "Tramontina"
  "modelo"             : string — ex: "LTD12", "Galaxy S21", null
  "tipo_produto"       : string — ex: "máquina de lavar", "geladeira"
  "capacidade_volume"  : string — ex: "12kg", "500ml", "2000ml", null
  "voltagem"           : string — "110v", "220v" ou "bivolt"; null se ausente
  "cor"                : string — ex: "branco", "preto", null
  "material"           : string — ex: "inox", "plástico", null
  "quantidade"         : string — ex: "kit 3 peças", "par", null
  "outros_atributos"   : string — outros atributos relevantes, null se não houver

══════════════════════════════════════════════════════════════════
REGRAS DE NORMALIZAÇÃO (unidades e formato)
══════════════════════════════════════════════════════════════════

1. Volume → converter SEMPRE para ml sem espaço:
   "1/2 litro", "meio litro", "0,5 litro" → "500ml"
   "1 litro", "1l" → "1000ml"   |   "1,5 litro" → "1500ml"
   "2 litros" → "2000ml"

2. Voltagem → formato padronizado:
   "110 v", "110 volts", "110V" → "110v"
   "220 v", "220 volts" → "220v"
   "bivolt", "bi-volt", "Bi Volt" → "bivolt"

3. Peso/capacidade → sem espaço:
   "12 kilos", "12 kg", "12KG" → "12kg"
   "500 g", "500gr", "500 gramas" → "500g"

4. Texto em minúsculo para todos os valores, exceto marcas (1ª letra maiúscula).

══════════════════════════════════════════════════════════════════
REGRAS DE DESAMBIGUAÇÃO SEMÂNTICA
══════════════════════════════════════════════════════════════════

5. SINÔNIMOS DE TIPO — use sempre a forma CANÔNICA:
   "lavadora" = "máquina de lavar" → use "máquina de lavar"
   "geladeira" = "refrigerador" → use "geladeira"
   "liq." = "liquidificador" → use "liquidificador"
   "TV" = "televisão" = "televisor" → use "televisão"
   "notebook" = "laptop" → use "notebook"
   "celular" = "smartphone" = "telefone celular" → use "celular"
   "fone" = "fone de ouvido" = "headphone" = "earphone" → use "fone de ouvido"

6. MARCA — unifique variações gráficas da mesma marca:
   "Coca-Cola" = "Coca Cola" = "CocaCola" = "coca cola" → "Coca-Cola"
   Sempre capitalize a 1ª letra de cada palavra da marca.

7. Se o título contém apenas nome genérico sem marca, use null para marca.
   NÃO invente marcas.

8. Se um atributo é ambíguo, prefira a interpretação mais provável para
   o contexto do produto.


══════════════════════════════════════════════════════════════════
FORMATO DE SAÍDA (BATCH)
══════════════════════════════════════════════════════════════════

Retorne SOMENTE um array JSON válido. Sem texto extra, sem markdown.
Cada objeto deve conter o campo "idx" com o índice recebido.
"""

### Função de extração — modo batch

A função agora recebe uma lista de títulos e retorna uma lista de JSONs,
enviando todos em uma única chamada de API.

In [21]:
def extract_batch_llm(
    titles: list[str],
    categories: list[str],
    client: genai.GenerativeModel,
    start_idx: int = 0,
) -> list[dict]:
    """
    Envia múltiplos títulos em uma chamada (BATCHER-inspired).
    Inclui categoria como contexto (Narayan et al., 2022 — attribute selection).
    """
    lines = []
    for i, (title, cat) in enumerate(zip(titles, categories)):
        lines.append(f'[{start_idx + i}] Categoria: "{cat}" | Título: "{title}"')
    user_prompt = "\n".join(lines)

    try:
        response = client.generate_content(user_prompt)
        raw_text = response.text.strip()

        # Limpa markdown se houver
        if raw_text.startswith("```"):
            raw_text = re.sub(r"^```(?:json)?", "", raw_text).rstrip("`").strip()

        parsed = json.loads(raw_text)

        # Se o LLM retornou um dict único em vez de array, embrulhar
        if isinstance(parsed, dict):
            parsed = [parsed]

        return parsed

    except json.JSONDecodeError:
        # Fallback: tentar extrair cada {...} individualmente
        matches = re.findall(r"\{[^{}]+\}", raw_text, re.DOTALL)
        results = []
        for m in matches:
            try:
                results.append(json.loads(m))
            except json.JSONDecodeError:
                continue
        if results:
            return results

        logger.warning("Sem JSON extraível do batch. Texto: %s", repr(raw_text[:200]))
        return [{} for _ in titles]

    except ValueError as e:
        logger.warning("Resposta bloqueada para batch: %s", e)
        return [{} for _ in titles]

    except Exception as e:
        logger.error("Erro LLM batch: %s", e)
        return [{} for _ in titles]

## Passo 4 — Limpeza Determinística (Camada Simbólica)

> Mesma lógica do pipeline v1, mantida como safety net

In [10]:
_RE_SPACES     = re.compile(r"\s+")
_RE_VOLUME_ML  = re.compile(r"(\d+(?:[.,]\d+)?)\s*ml", re.I)
_RE_VOLUME_L   = re.compile(r"(\d+(?:[.,]\d+)?)\s*l(?:itro[s]?)?(?=\b|$)", re.I)
_RE_WEIGHT_KG  = re.compile(r"(\d+(?:[.,]\d+)?)\s*k(?:g|ilos?|ilogramas?)", re.I)
_RE_WEIGHT_G   = re.compile(r"(\d+(?:[.,]\d+)?)\s*g(?:r|ramas?)?(?=\b|$)", re.I)
_RE_VOLTAGE    = re.compile(r"(\d+)\s*v(?:olts?)?(?=\b|$)", re.I)
_RE_BIVOLT     = re.compile(r"bi[\s-]?volt", re.I)


def _clean_str(value: str | None) -> str:
    if not value or not isinstance(value, str):
        return ""
    return _RE_SPACES.sub(" ", value.strip()).lower()


def _normalize_volume(value: str) -> str:
    if not value:
        return value
    value = value.replace(",", ".")
    m = _RE_VOLUME_ML.search(value)
    if m:
        qty = float(m.group(1))
        return f"{int(qty)}ml" if qty == int(qty) else f"{qty}ml"
    m = _RE_VOLUME_L.search(value)
    if m:
        return f"{int(float(m.group(1)) * 1000)}ml"
    return value


def _normalize_weight(value: str) -> str:
    if not value:
        return value
    value = value.replace(",", ".")
    m = _RE_WEIGHT_KG.search(value)
    if m:
        qty = float(m.group(1))
        return f"{int(qty)}kg" if qty == int(qty) else f"{qty}kg"
    m = _RE_WEIGHT_G.search(value)
    if m:
        qty = float(m.group(1))
        return f"{int(qty)}g" if qty == int(qty) else f"{qty}g"
    return value


def _normalize_voltage(value: str) -> str:
    if not value:
        return value
    if _RE_BIVOLT.search(value):
        return "bivolt"
    m = _RE_VOLTAGE.search(value)
    return f"{m.group(1)}v" if m else value


def _normalize_capacity(value: str) -> str:
    if not value:
        return value
    v = value.lower()
    if any(u in v for u in ["ml", "litro", " l", "l "]):
        return _normalize_volume(v)
    if any(u in v for u in ["kg", "kilo", "grama", " g", "g "]):
        return _normalize_weight(v)
    return _clean_str(value)


def clean_llm_output(raw_json: dict) -> dict:
    if not raw_json:
        return {}
    cleaned = {}
    for key, value in raw_json.items():
        if key == "idx":
            continue  # campo de controle do batch
        if value is None or value == "null":
            cleaned[key] = None
            continue
        s = str(value).strip()
        if key == "marca":
            cleaned[key] = _RE_SPACES.sub(" ", s).title() or None
        elif key == "voltagem":
            cleaned[key] = _normalize_voltage(_clean_str(s)) or None
        elif key == "capacidade_volume":
            cleaned[key] = _normalize_capacity(_clean_str(s)) or None
        else:
            cleaned[key] = _clean_str(s) or None
    return cleaned


def build_canonical_name(cleaned_json: dict) -> str:
    ORDER = [
        "tipo_produto", "marca", "modelo", "capacidade_volume",
        "voltagem", "cor", "material", "quantidade", "outros_atributos",
    ]
    parts = [
        str(cleaned_json[f])
        for f in ORDER
        if cleaned_json.get(f) not in (None, "", "null")
    ]
    return " | ".join(parts)

## Passo 5 — Orquestração (Batch) e Exportação

In [22]:
os.makedirs("output", exist_ok=True)

genai.configure(api_key=API_KEY)
client = genai.GenerativeModel(
    model_name=LLM_MODEL,
    system_instruction=SYSTEM_PROMPT,
    generation_config=genai.GenerationConfig(
        response_mime_type="application/json",
        temperature=TEMPERATURE,
        max_output_tokens=2048,  # aumentado para suportar batch
    ),
)

In [23]:
# ── Loop principal com batch prompting ─────────────────────────
titles_pre = df_sample["titulo_pre_normalizado"].tolist()
categories = df_sample["category"].tolist()
n = len(titles_pre)

all_raw_jsons  = [None] * n
all_cleaned    = [None] * n
all_canonical  = [None] * n

for batch_start in tqdm(range(0, n, BATCH_SIZE), desc="Batches", unit="batch"):
    batch_end    = min(batch_start + BATCH_SIZE, n)
    batch_titles = titles_pre[batch_start:batch_end]
    batch_cats   = categories[batch_start:batch_end]

    raw_list = extract_batch_llm(batch_titles, batch_cats, client, start_idx=batch_start)

    # Alinhar resultados com os títulos do batch
    for local_i in range(batch_end - batch_start):
        global_i = batch_start + local_i

        # Tentar encontrar o JSON correspondente pelo idx
        matched = None
        for r in raw_list:
            if isinstance(r, dict) and r.get("idx") == global_i:
                matched = r
                break

        # Fallback: usar posição se idx não bate
        if matched is None and local_i < len(raw_list):
            matched = raw_list[local_i] if isinstance(raw_list[local_i], dict) else {}

        if matched is None:
            matched = {}

        cleaned = clean_llm_output(matched)
        canon   = build_canonical_name(cleaned)

        all_raw_jsons[global_i] = matched
        all_cleaned[global_i]   = cleaned
        all_canonical[global_i] = canon

    time.sleep(2)  # rate limit gentil (menor que v1 graças ao batch)

print(f"Processamento concluído: {n} títulos em {(n + BATCH_SIZE - 1) // BATCH_SIZE} batches")

Batches: 100%|██████████| 40/40 [03:32<00:00,  5.31s/batch]

Processamento concluído: 200 títulos em 40 batches


In [24]:
df_result = pd.DataFrame({
    "titulo_original":     df_sample["title"].values,
    "titulo_pre_norm":     df_sample["titulo_pre_normalizado"].values,
    "categoria_original":  df_sample["category"].values,
    "json_llm_bruto":      [json.dumps(r, ensure_ascii=False) for r in all_raw_jsons],
    "json_limpo_python":   [json.dumps(c, ensure_ascii=False) for c in all_cleaned],
    "nome_canonico_final": all_canonical,
})

df_result.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"Exportado: {OUTPUT_PATH}")
df_result.head(10)

Exportado: output/resultado_pipeline_v2.csv


,titulo_original,titulo_pre_norm,categoria_original,json_llm_bruto,json_limpo_python,nome_canonico_final
0,2 Unidades Caixa Transporte Para Cães E Gatos ...,2 Unidades Caixa Transporte Para Cães E Gatos ...,DOG_CARRIERS_AND_CARRYING_BAGS,"{""idx"": 0, ""marca"": ""Pet Grillo"", ""modelo"": ""n...","{""marca"": ""Pet Grillo"", ""modelo"": ""n°2"", ""tipo...",caixa de transporte | Pet Grillo | n°2 | 2 uni...
1,Ferro De Passar Dimar Elétrico Funcionando Rar...,Ferro De Passar Dimar Elétrico Funcionando Rar...,IRONS,"{""idx"": 1, ""marca"": ""Dimar"", ""modelo"": null, ""...","{""marca"": ""Dimar"", ""modelo"": null, ""tipo_produ...","ferro de passar | Dimar | elétrico, raríssimo"
2,Bateria Carregador Celular Samsung Young 2 130,Bateria Carregador Celular Samsung Young 2 130,CELLPHONE_BATTERIES,"{""idx"": 2, ""marca"": ""Samsung"", ""modelo"": ""youn...","{""marca"": ""Samsung"", ""modelo"": ""young 2 130"", ...",bateria | Samsung | young 2 130 | carregador
3,Vassoura Elétrica Portátil Aspirador Pó Vermel...,Vassoura Elétrica Portátil Aspirador Pó Vermel...,ELECTRIC_BROOMS,"{""idx"": 3, ""marca"": ""Philco"", ""modelo"": null, ...","{""marca"": ""Philco"", ""modelo"": null, ""tipo_prod...",vassoura elétrica | Philco | 220v | vermelho |...
4,Depilador Tipo Laser Yes Finishing Touch Envio...,Depilador Tipo Laser Yes Finishing Touch Envio...,EPILATORS,"{""idx"": 4, ""marca"": ""Yes Finishing Touch"", ""mo...","{""marca"": ""Yes Finishing Touch"", ""modelo"": nul...",depilador | Yes Finishing Touch | tipo laser
5,Coxim Diant Motor L200 4x4 95/ Mobensani ...,Coxim Diant Motor L200 4x4 95/ Mobensani,AUTOMOTIVE_ENGINE_MOUNTS,"{""idx"": 5, ""marca"": ""Mobensani"", ""modelo"": ""L2...","{""marca"": ""Mobensani"", ""modelo"": ""l200 4x4 95/...",coxim de motor | Mobensani | l200 4x4 95/ | di...
6,Álbum De Fotos Floral De Couro Ecol.200 Fotos ...,Álbum De Fotos Floral De Couro Ecol.200 Fotos ...,PHOTO_ALBUMS,"{""idx"": 6, ""marca"": null, ""modelo"": null, ""tip...","{""marca"": null, ""modelo"": null, ""tipo_produto""...",álbum de fotos | preto | couro ecológico | 200...
7,Jaqueta Couro Personalizada Motociclista C/ Pr...,Jaqueta Couro Personalizada Motociclista C/ Pr...,MOTORCYCLE_JACKETS,"{""idx"": 7, ""marca"": null, ""modelo"": null, ""tip...","{""marca"": null, ""modelo"": null, ""tipo_produto""...","jaqueta | couro | personalizada, motociclista,..."
8,Café Poesia Torrado Em Grão 1 Kg .............,Café Poesia Torrado Em Grão 1 Kg ............ ...,GROUND_AND_WHOLE_BEANS_COFFEE,"{""idx"": 8, ""marca"": ""Poesia"", ""modelo"": null, ...","{""marca"": ""Poesia"", ""modelo"": null, ""tipo_prod...","café | Poesia | 1kg | torrado em grão, exportação"
9,Tampa Traseira Passat 2010,Tampa Traseira Passat 2010,AUTOMOTIVE_TRUNK_LIDS,"{""idx"": 9, ""marca"": ""Volkswagen"", ""modelo"": ""P...","{""marca"": ""Volkswagen"", ""modelo"": ""passat 2010...",tampa traseira | Volkswagen | passat 2010


## Passo 6 — Avaliação Estruturada

> Módulo `avaliacao_pipeline.py` rodado standalone no repositório.
> Aqui importamos e executamos diretamente sobre `df_result`.

In [25]:
from eval import gerar_relatorio

relatorio = gerar_relatorio(df_result, verbose=True)

  1. QUALIDADE DA EXTRAÇÃO JSON
  Total de registros:        200
  JSONs válidos com dados:   200  (100.0%)
  JSONs vazios/erro:         0
  JSONs todos nulos:         0
  Taxa de falha total:       0.0%

  2. COMPLETUDE DE ATRIBUTOS
  tipo_produto           ████████████████████  100.0%  (200/200)
  outros_atributos       █████████████████░░░   85.0%  (170/200)
  marca                  ███████████░░░░░░░░░   58.5%  (117/200)
  modelo                 ███████████░░░░░░░░░   57.0%  (114/200)
  quantidade             █████░░░░░░░░░░░░░░░   27.0%  (54/200)
  material               ████░░░░░░░░░░░░░░░░   21.0%  (42/200)
  cor                    ███░░░░░░░░░░░░░░░░░   19.5%  (39/200)
  capacidade_volume      ██░░░░░░░░░░░░░░░░░░   11.5%  (23/200)  ⚠️ CRÍTICO
  voltagem               █░░░░░░░░░░░░░░░░░░░    8.5%  (17/200)

  3. ANÁLISE DE AGRUPAMENTO
  Registros com nome:        200
  Nomes canônicos únicos:    200
  Taxa de redução:           0.0%
  Singletons:                200  (100.0% dos

## Verificação Rápida — Exemplos do Pipeline v2

In [27]:
test_titles = [
    ("COCA COLA 500 ml",                         "Refrigerantes"),
    ("coca-cola 500ml",                           "Refrigerantes"),
    ("Coca Cola 0,5 litro",                       "Refrigerantes"),
    ("CocaCola 1/2 litro",                        "Refrigerantes"),
    ("Coca Cola meio litro",                      "Refrigerantes"),
    ("Lavadora Electrolux 12 Kilos 220v Branca",  "Eletrodomésticos"),
    ("Maquina De Lavar Electrolux 12kg 220 Volts","Eletrodomésticos"),
    ("Liq. Mondial Turbo Inox 1000w 110v",        "Eletrodomésticos"),
    ("Kit 3 Pçs Panela Tramontina Inox",          "Cozinha"),
    ("Smart TV Samsung 55 4K Crystal Bivolt",     "Eletrônicos"),
]

# Pré-normalizar
pre_normed = [(pre_normalize_title(t), c) for t, c in test_titles]

print("Pré-normalização:")
print("-" * 65)
for (orig, _), (normed, _) in zip(test_titles, pre_normed):
    flag = " ←" if orig != normed else ""
    print(f"  {orig:<50s} → {normed}{flag}")

# Processar via LLM (batch)
titles_batch = [t for t, _ in pre_normed]
cats_batch   = [c for _, c in pre_normed]
results = extract_batch_llm(titles_batch, cats_batch, client, start_idx=0)

print()
print("Resultados:")
print("-" * 65)
for i, (orig_title, _) in enumerate(test_titles):
    raw = results[i] if i < len(results) else {}
    cleaned = clean_llm_output(raw)
    canon   = build_canonical_name(cleaned)
    print(f"  Título:    {orig_title}")
    print(f"  Canônico:  {canon}")
    print()

Pré-normalização:
-----------------------------------------------------------------
  COCA COLA 500 ml                                   → COCA COLA 500 ml
  coca-cola 500ml                                    → coca-cola 500ml
  Coca Cola 0,5 litro                                → Coca Cola 500ml ←
  CocaCola 1/2 litro                                 → CocaCola 500ml ←
  Coca Cola meio litro                               → Coca Cola 500ml ←
  Lavadora Electrolux 12 Kilos 220v Branca           → Lavadora Electrolux 12kg 220v Branca ←
  Maquina De Lavar Electrolux 12kg 220 Volts         → Maquina De Lavar Electrolux 12kg 220v ←
  Liq. Mondial Turbo Inox 1000w 110v                 → Liq. Mondial Turbo Inox 1000w 110v
  Kit 3 Pçs Panela Tramontina Inox                   → Kit 3 Pçs Panela Tramontina Inox
  Smart TV Samsung 55 4K Crystal Bivolt              → Smart TV Samsung 55 4kg Crystal bivolt ←

Resultados:
-----------------------------------------------------------------
  Título:    